In [ ]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import subprocess
import threading
import sys
import time
import os
import nbformat
from nbclient import NotebookClient
import socket

class NAOLauncher:
    def __init__(self, root):
        self.root = root
        self.root.title("Fitness Coach with Robot")
        
        # Configure main window - increased size
        self.root.geometry("800x700")
        
        # Configure style for larger fonts
        self.style = ttk.Style()
        self.style.configure('Large.TLabel', font=('Arial', 12))
        self.style.configure('Large.TButton', font=('Arial', 12))
        self.style.configure('Header.TLabel', font=('Arial', 14, 'bold'))
        self.style.configure('Large.TLabelframe.Label', font=('Arial', 12, 'bold'))
        
        # Initialize process handlers
        self.server_process = None
        self.detection_process = None
        self.choregraphe_process = None
        
        # Track individual component states
        self.choregraphe_running = False
        self.server_running = False
        self.detection_running = False
        
        # Paths
        self.choregraphe_path = r"C:\Program Files (x86)\Softbank Robotics\Choregraphe Suite 2.8\bin\choregraphe_launcher.exe"
        self.behavior_file = None
        
        # Robot IP
        self.robot_ip = tk.StringVar()
        
        self.create_widgets()
        
    def create_widgets(self):
        # Main frame with increased padding
        main_frame = ttk.Frame(self.root, padding="20")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Title
        title_label = ttk.Label(main_frame, text="Fitness Coach with Robot", style='Header.TLabel')
        title_label.grid(row=0, column=0, pady=(0, 20))
        
        # Choregraphe section
        chore_frame = ttk.LabelFrame(main_frame, text="Choregraphe", padding="15", style='Large.TLabelframe')
        chore_frame.grid(row=1, column=0, pady=10, sticky=(tk.W, tk.E))
        
        # Configure grid column weights
        chore_frame.grid_columnconfigure(0, weight=3)
        chore_frame.grid_columnconfigure(1, weight=1)
        
        path_label = ttk.Label(chore_frame, text="Project Path:", style='Large.TLabel')
        path_label.grid(row=0, column=0, padx=5, sticky=tk.W)
        
        self.behavior_path = ttk.Entry(chore_frame, width=50, font=('Arial', 11))
        self.behavior_path.grid(row=1, column=0, padx=5, pady=5, sticky=tk.W)
        
        browse_btn = ttk.Button(chore_frame, text="Browse", command=self.browse_behavior, style='Large.TButton')
        browse_btn.grid(row=1, column=1, padx=10, pady=5)
        
        self.chore_btn = ttk.Button(chore_frame, text="Start Choregraphe", 
                                   command=self.toggle_choregraphe, style='Large.TButton')
        self.chore_btn.grid(row=2, column=0, pady=10, sticky=tk.W)
        
        self.chore_status = ttk.Label(chore_frame, text="Status: Not running", 
                                     foreground="red", style='Large.TLabel')
        self.chore_status.grid(row=2, column=1, pady=10)
        
        # Server section
        server_frame = ttk.LabelFrame(main_frame, text="Server Configuration", 
                                    padding="15", style='Large.TLabelframe')
        server_frame.grid(row=2, column=0, pady=10, sticky=(tk.W, tk.E))
        
        # IP input field
        ip_frame = ttk.Frame(server_frame)
        ip_frame.grid(row=0, column=0, columnspan=2, pady=10)
        
        ttk.Label(ip_frame, text="Robot IP Address:", style='Large.TLabel').grid(row=0, column=0, padx=5)
        self.ip_entry = ttk.Entry(ip_frame, textvariable=self.robot_ip, width=20, font=('Arial', 11))
        self.ip_entry.grid(row=0, column=1, padx=10)
        
        # Server controls
        self.server_btn = ttk.Button(server_frame, text="Start Server", 
                                    command=self.toggle_server, style='Large.TButton')
        self.server_btn.grid(row=1, column=0, pady=10, sticky=tk.W)
        
        self.server_status = ttk.Label(server_frame, text="Status: Stopped", 
                                      foreground="red", style='Large.TLabel')
        self.server_status.grid(row=1, column=1, pady=10)
        
        # Detection App section
        detection_frame = ttk.LabelFrame(main_frame, text="Detection Application", 
                                       padding="15", style='Large.TLabelframe')
        detection_frame.grid(row=3, column=0, pady=10, sticky=(tk.W, tk.E))
        
        self.detection_btn = ttk.Button(detection_frame, text="Start Detection", 
                                      command=self.toggle_detection, style='Large.TButton')
        self.detection_btn.grid(row=0, column=0, pady=10, sticky=tk.W)
        
        self.detection_status = ttk.Label(detection_frame, text="Status: Stopped", 
                                        foreground="red", style='Large.TLabel')
        self.detection_status.grid(row=0, column=1, pady=10)
        
        # Status messages
        status_frame = ttk.LabelFrame(main_frame, text="System Log", padding="15", style='Large.TLabelframe')
        status_frame.grid(row=4, column=0, pady=10, sticky=(tk.W, tk.E))
        
        self.status_text = tk.Text(status_frame, height=5, width=80, font=('Consolas', 11))
        self.status_text.grid(row=0, column=0, pady=5)
        
        # Add scrollbar to status text
        scrollbar = ttk.Scrollbar(status_frame, orient="vertical", command=self.status_text.yview)
        scrollbar.grid(row=0, column=1, sticky=(tk.N, tk.S))
        self.status_text.configure(yscrollcommand=scrollbar.set)

    def validate_ip(self, ip_address):
        try:
            socket.inet_aton(ip_address)
            return True
        except socket.error:
            return False
        
    def browse_behavior(self):
        filename = filedialog.askopenfilename(
            title="Select Choregraphe Project",
            filetypes=[("Choregraphe project", "*.pml"), ("All files", "*.*")]
        )
        if filename:
            self.behavior_file = filename
            self.behavior_path.delete(0, tk.END)
            self.behavior_path.insert(0, filename)
            self.log_message(f"Selected Choregraphe project: {filename}")
    
    def toggle_choregraphe(self):
        if not self.choregraphe_running:
            self.start_choregraphe()
        else:
            self.stop_choregraphe()
    
    def toggle_server(self):
        if not self.server_running:
            # Validate IP before starting server
            ip = self.robot_ip.get().strip()
            if not ip:
                messagebox.showerror("Error", "Please enter the robot's IP address")
                return
            if not self.validate_ip(ip):
                messagebox.showerror("Error", "Invalid IP address format")
                return
            self.start_server()
        else:
            self.stop_server()
    
    def toggle_detection(self):
        if not self.detection_running:
            self.start_detection_app()
        else:
            self.stop_detection_app()
    
    def start_choregraphe(self):
        try:
            cmd = [self.choregraphe_path]
            if self.behavior_file:
                cmd.append(self.behavior_file)
                
            self.choregraphe_process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                universal_newlines=True
            )
            
            self.choregraphe_running = True
            self.chore_status.configure(text="Status: Running", foreground="green")
            self.chore_btn.configure(text="Stop Choregraphe")
            self.log_message("Choregraphe started successfully")
            
            threading.Thread(target=self.monitor_output, 
                           args=(self.choregraphe_process.stdout, "Choregraphe"),
                           daemon=True).start()
            
        except Exception as e:
            self.log_message(f"Error starting Choregraphe: {str(e)}")
            self.chore_status.configure(text="Status: Error", foreground="red")
    
    def start_server(self):
        try:
            python27_path = r"C:\Python27\python.exe"
            server_script = "nao_stream_server.py"
            
            # Pass the IP address as an environment variable
            env = os.environ.copy()
            env['NAO_IP'] = self.robot_ip.get().strip()
            
            self.server_process = subprocess.Popen(
                [python27_path, server_script, self.robot_ip.get().strip()],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                universal_newlines=True,
                env=env
            )
            
            self.server_running = True
            self.server_status.configure(text="Status: Running", foreground="green")
            self.server_btn.configure(text="Stop Server")
            self.log_message(f"NAO server started successfully with IP: {env['NAO_IP']}")
            
            threading.Thread(target=self.monitor_output, 
                           args=(self.server_process.stdout, "Server"),
                           daemon=True).start()
            
        except Exception as e:
            self.log_message(f"Error starting server: {str(e)}")
            self.server_status.configure(text="Status: Error", foreground="red")
    
    def start_detection_app(self):
        try:
            with open("nao_stream_client.ipynb") as f:
                notebook = nbformat.read(f, as_version=4)
            
            client = NotebookClient(notebook)
            client.execute()
            
            self.detection_running = True
            self.detection_status.configure(text="Status: Running", foreground="green")
            self.detection_btn.configure(text="Stop Detection")
            self.log_message("Detection app started successfully")
            
        except Exception as e:
            self.log_message(f"Error starting detection app: {str(e)}")
            self.detection_status.configure(text="Status: Error", foreground="red")
    
    def stop_choregraphe(self):
        if self.choregraphe_process:
            try:
                self.choregraphe_process.terminate()
                self.choregraphe_running = False
                self.chore_status.configure(text="Status: Stopped", foreground="red")
                self.chore_btn.configure(text="Start Choregraphe")
                self.log_message("Choregraphe stopped")
            except Exception as e:
                self.log_message(f"Error stopping Choregraphe: {str(e)}")
    
    def stop_server(self):
        if self.server_process:
            try:
                self.server_process.terminate()
                self.server_running = False
                self.server_status.configure(text="Status: Stopped", foreground="red")
                self.server_btn.configure(text="Start Server")
                self.log_message("Server stopped")
            except Exception as e:
                self.log_message(f"Error stopping server: {str(e)}")
    
    def stop_detection_app(self):
        try:
            self.detection_running = False
            self.detection_status.configure(text="Status: Stopped", foreground="red")
            self.detection_btn.configure(text="Start Detection")
            self.log_message("Detection app stopped")
        except Exception as e:
            self.log_message(f"Error stopping detection app: {str(e)}")
    
    def monitor_output(self, pipe, prefix):
        for line in iter(pipe.readline, ''):
            self.log_message(f"{prefix}: {line.strip()}")
    
    def log_message(self, message):
        self.status_text.insert(tk.END, f"{message}\n")
        self.status_text.see(tk.END)
    
    def on_closing(self):
        if self.choregraphe_running:
            self.stop_choregraphe()
        if self.server_running:
            self.stop_server()
        if self.detection_running:
            self.stop_detection_app()
        self.root.destroy()

def main():
    root = tk.Tk()
    app = NAOLauncher(root)
    root.protocol("WM_DELETE_WINDOW", app.on_closing)
    root.mainloop()

if __name__ == "__main__":
    main()